# 🎬 Kanji: ảnh câu chuyện → video hoạt hình → GIF 5 giây (cả bộ 2.136 chữ)

Ảnh câu chuyện (tranh bên trái) + câu mô tả chuyển động → clip AI, rồi ghép **câu chuyện hiện từng cụm + chữ Kanji to** thành GIF 5 giây.

- Câu mô tả **tự tạo** từ nghĩa + câu chuyện tiếng Anh (`kanji-en.js` trên trang web) — mọi thẻ có câu chuyện (~2.075 thẻ). Muốn mô tả riêng thẻ nào thì ghi vào `CUSTOM` ở ô 3.
- Kết quả lưu vào **Google Drive** `MyDrive/kanji_anim/` (`mp4/`, `gif/`). Máy bị ngắt → mở lại, **Run all** là làm tiếp từ thẻ chưa có.
- Mỗi lần chạy làm tối đa `PER_RUN` thẻ. T4 ≈ 4–6 phút/thẻ → cả bộ ≈ 150–200 giờ GPU, chia nhiều phiên.

| `ENGINE` | Model | GPU | Ghi chú |
|---|---|---|---|
| `"ltx"` | **LTX-Video 0.9.1** (2B) | **T4 free** chạy được | 704×480 trên T4, 768×512 trên GPU lớn |
| `"wan"` | **Wan2.2-TI2V-5B** | nên **L4 / A100** | đẹp hơn; 1280×704 |

*Runtime → Change runtime type →* chọn GPU → **Run all**.

In [ ]:
import torch
assert torch.cuda.is_available(), f"⚠️ Chưa có GPU (torch {torch.__version__}). Runtime → Change runtime type → T4 GPU, rồi chạy lại từ đầu."
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Chọn model

In [ ]:
ENGINE = "ltx"   # "ltx" (nhẹ, T4 free) hoặc "wan" (đẹp hơn, cần L4/A100)

## 2. Cài đặt + tải model (LTX ~25 GB / Wan ~20 GB, mất 5–10 phút mỗi phiên)

In [ ]:
!apt-get -qq update > /dev/null; apt-get -qq install -y fonts-noto-cjk fonts-noto-core > /dev/null   # font chữ Hán + tiếng Việt cho GIF
!pip -q install "imageio[ffmpeg]"
if ENGINE == "wan":
    %cd /content
    ![ -d Wan2.2 ] || git clone -q https://github.com/Wan-Video/Wan2.2.git
    %cd /content/Wan2.2
    # cài mọi thứ trừ flash_attn (tự biên dịch trên Colab rất lâu / hay lỗi)
    !grep -v -i "flash" requirements.txt > /tmp/req.txt && pip -q install -r /tmp/req.txt
    !pip -q install flash-attn --no-build-isolation --only-binary=:all: || echo "⚠️ Không cài được flash_attn — vẫn chạy được, chỉ chậm hơn"
    !pip -q install "huggingface_hub[cli]"
    !huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir /content/Wan2.2-TI2V-5B
else:
    !pip -q install -U diffusers transformers accelerate
    !huggingface-cli download a-r-r-o-w/LTX-Video-0.9.1-diffusers --local-dir /content/LTX-Video-0.9.1

## 3. Cấu hình

- `ONLY`: để trống `[]` = làm tất cả; hoặc ghi số thẻ, vd `[17, 542]` hay `list(range(1, 101))`.
- `CUSTOM`: câu mô tả viết tay (tiếng Anh) cho thẻ nào muốn chuyển động cụ thể hơn; thẻ còn lại tự tạo.
- `PER_RUN`: số thẻ tối đa mỗi lần chạy (Colab free hay ngắt sau vài giờ).

In [ ]:
import os, torch
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/kanji_anim"          # mp4 + gif lưu ở đây, chạy tiếp được
os.makedirs(f"{OUT}/mp4", exist_ok=True); os.makedirs(f"{OUT}/gif", exist_ok=True)

ONLY = []        # [] = tất cả thẻ có câu chuyện
PER_RUN = 60     # số thẻ mỗi lần chạy
CUSTOM = {
    17:   "A boy with arms spread out keeps walking forward step by step on brown ground, his body leaning as if something heavy presses on him",
    542:  "A tall young woman stands still while a small girl in a green dress holds her hand and looks up at her, the small black arrow gently pulses",
    2095: "A bald Buddhist monk with closed eyes gently rubs his shiny bald head with one hand and smiles peacefully, small golden sparkles twinkle above his head",
}
STYLE = "simple hand-drawn cartoon line art, flat colors, white background, static camera, gentle smooth motion, keep the same drawing style"
NEG = "worst quality, blurry, distorted, deformed face, extra limbs, jittery, text, letters, watermark, photo realistic"
FAST = True      # nhanh hơn, chất lượng thấp hơn chút
SITE = "https://testervippro.github.io/Giai-Thich-Ngu-Phap-Tieng-Anh/kanji-viewer/"

SMALL_GPU = torch.cuda.get_device_properties(0).total_memory < 20e9   # T4 15 GB
if ENGINE == "wan":
    W, H = 1280, 704
    FRAMES, STEPS = (49, 20) if FAST else (121, 50)   # 4n+1
elif SMALL_GPU:   # T4: không có flash-attention cho bf16 -> giảm khung + số frame để không hết bộ nhớ GPU
    W, H = 704, 480
    FRAMES, STEPS = (65, 30) if FAST else (81, 40)    # 8n+1; GIF chạy qua-lại nên vẫn đủ 5 s
else:
    W, H = 768, 512
    FRAMES, STEPS = (97, 30) if FAST else (121, 50)   # 8n+1, 24 fps
print(ENGINE, W, H, FRAMES, STEPS, "| lưu vào", OUT)

## 4. Dữ liệu thẻ + câu mô tả + hàm chuẩn bị ảnh + hàm ghép GIF

In [ ]:
import os, io, re, json, glob, subprocess, urllib.request
import numpy as np
import imageio.v3 as iio
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as IPImage, display

def js_value(name, var):
    s = urllib.request.urlopen(SITE + name).read().decode("utf-8")
    return json.JSONDecoder().raw_decode(s, re.search(rf"window\.{var}\s*=\s*", s).end())[0]

KJ = {r[0]: r for r in js_value("kanji-game-data.js", "KANJI")}   # [thẻ, chữ, Hán Việt, nghĩa, JLPT, câu chuyện, ...]
EN = js_value("kanji-en.js", "KANJI_EN")                          # {chữ: [nghĩa EN, câu chuyện EN]}

def prompt(no):
    if no in CUSTOM: return f"{CUSTOM[no]}. {STYLE}"
    mean, story = EN.get(KJ[no][1], ["", ""])
    return (f"A cartoon illustration of a little story: {story} "
            f"The people, animals and objects in the picture come alive and move gently and naturally. {STYLE}")

ALL = [no for no in sorted(KJ) if KJ[no][5].strip()]
CARDS = [no for no in (ONLY or ALL) if no in KJ and KJ[no][5].strip()]
done = lambda no: os.path.exists(f"{OUT}/gif/{no:04d}.gif")
TODO = [no for no in CARDS if not done(no)][:PER_RUN]
print(f"Thẻ có câu chuyện: {len(ALL)} | chọn: {len(CARDS)} | đã xong: {sum(map(done, CARDS))} | lần này làm: {len(TODO)}")
if TODO: print("Ví dụ câu mô tả:", TODO[0], prompt(TODO[0]))

# ---- ảnh: cắt phần tranh (bên trái ảnh câu chuyện), đặt giữa khung nền trắng ----
os.makedirs("/content/in", exist_ok=True)
def picture_part(im):
    """Tranh ở bên trái: khối có mực đầu tiên, kết thúc ở khoảng cột trắng >= 25 px."""
    a = np.asarray(im.convert("RGB")).astype(int); h, w, _ = a.shape
    cols = ((a.min(2) < 225) | (((a.max(2) - a.min(2)) > 35) & (a.min(2) < 245))).mean(0)
    x0 = int(np.argmax(cols > 0.02)); x1, gap = x0, 0
    for x in range(x0, w):
        if cols[x] > 0.005: x1, gap = x, 0
        else:
            gap += 1
            if gap >= 25 and x - x0 > 100: break
    return im.crop((0, 0, x1 + 8, h))

def prep_image(no):
    path = f"/content/in/{no:04d}.png"
    if os.path.exists(path): return path
    raw = urllib.request.urlopen(SITE + f"kanji_story/{no:04d}.webp").read()
    pic = picture_part(Image.open(io.BytesIO(raw)).convert("RGB"))
    s = min(W * .9 / pic.width, H * .9 / pic.height)
    p = pic.resize((round(pic.width * s), round(pic.height * s)), Image.LANCZOS)
    canvas = Image.new("RGB", (W, H), "white")
    canvas.paste(p, ((W - p.width) // 2, (H - p.height) // 2))
    canvas.save(path); return path

# ---- font: tự tìm, thiếu thì cài lại ----
def find_font(*names):
    for n in names:
        g = glob.glob(f"/usr/share/fonts/**/{n}", recursive=True)
        if g: return g[0]
if not find_font("NotoSansCJK-Bold.ttc", "NotoSansCJK*-Bold.otf"):
    print("Đang cài font Noto (≈1 phút)...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y fonts-noto-cjk fonts-noto-core", shell=True)
FV = find_font("NotoSans-BoldItalic.ttf", "NotoSans-Bold.ttf", "DejaVuSans-BoldOblique.ttf")
FM = find_font("NotoSans-Medium.ttf", "NotoSans-Regular.ttf", "DejaVuSans.ttf")
FC = find_font("NotoSansCJK-Bold.ttc", "NotoSansCJK*-Bold.otf", "NotoSansCJK-Regular.ttc")
assert FV and FM and FC, "Thiếu font — chạy: !apt-get update && apt-get install -y fonts-noto-cjk fonts-noto-core"

# ---- GIF 5 s: clip bên trái + câu chuyện hiện từng cụm (từ khoá xanh, chữ Hán đỏ) + chữ Kanji to ----
GW, GH, FPS, DUR = 800, 240, 12, 5.0
INK, TEAL, RED = (35, 35, 35), (30, 125, 175), (185, 35, 40)
cjk = lambda c: ord(c) >= 0x2E80

def chunks(story):
    out, buf = [], []
    for m in re.finditer(r"\(([^)]*)\)|[^\s()]+", story):
        if m.group(1) is not None: out.append((" ".join(buf), m.group(1).strip())); buf = []
        else: buf.append(m.group(0))
    if buf: out.append((" ".join(buf), None))
    return out

def words(ch):
    it = []
    for i, (txt, k) in enumerate(ch):
        ws = txt.split()
        for j, w in enumerate(ws):
            it.append((w, TEAL if k and j == len(ws) - 1 else INK, i, bool(re.fullmatch(r"[.,;:!?…]+", w))))
        if k: it.append(("(" + k + ")", RED, i, True))
    return it

def wlen(w, fv, sz): return sum(sz if cjk(c) else fv.getlength(c) for c in w)

def make_gif(no):
    _, k, hv, m, jlpt, story = KJ[no][:6]
    vid = iio.imread(f"{OUT}/mp4/{no:04d}.mp4")
    vid = np.concatenate([vid, vid[::-1]])                    # qua-lại cho mượt
    ph = GH - 24; pw = round(vid.shape[2] * ph / vid.shape[1])
    tx0, tx1 = pw + 30, GW - 16
    it = words(chunks(story)); nch = max(1, len(chunks(story)))
    for sz in (28, 25, 22, 20):
        fv, fc = ImageFont.truetype(FV, sz), ImageFont.truetype(FC, sz)
        sp = fv.getlength(" "); lines, cur, cw = [], [], 0
        for w in it:
            L = wlen(w[0], fv, sz); need = L + (0 if not cur else sp * (0.4 if w[3] else 1))
            if cur and cw + need > tx1 - tx0 and not w[3]: lines.append((cur, cw)); cur, cw, need = [], 0, L
            cur.append((w, L)); cw += need
        lines.append((cur, cw))
        if len(lines) * sz * 1.55 <= GH - 24: break
    frames = []
    for fi in range(int(FPS * DUR)):
        t = fi / FPS
        im = Image.new("RGB", (GW, GH), "white"); d = ImageDraw.Draw(im)
        v = Image.fromarray(vid[int(t * 24) % len(vid)]).resize((pw, ph), Image.LANCZOS)
        im.paste(v, (10, 12))
        fade = 1 - min(1, max(0, (t - 3.5) / 0.35))
        y = (GH - sz * 1.55 * len(lines)) / 2
        for ln, lw in lines:
            x = tx0 + (tx1 - tx0 - lw) / 2
            for n, ((w, col, ci, glue), L) in enumerate(ln):
                if n: x += (0 if re.fullmatch(r"[.,;:!?…]+", w) else sp * 0.4) if glue else sp
                a = min(1, max(0, (t - (0.6 + 2.8 * ci / nch)) / 0.3)) * fade
                if a > 0:
                    c2 = tuple(int(255 - (255 - c) * a) for c in col); xx = x
                    for ch_ in w:
                        if cjk(ch_):   # căn giữa nét chữ trong ô vuông rộng sz
                            il, _, ir, _ = fc.getmask(ch_).getbbox(); l0 = fc.getbbox(ch_)[0]
                            d.text((xx + (sz - (ir - il)) / 2 - l0 - il, y + 2), ch_, font=fc, fill=c2); xx += sz
                        else:
                            d.text((xx, y), ch_, font=fv, fill=c2); xx += fv.getlength(ch_)
                x += L
            y += sz * 1.55
        if t > 3.5:
            a = min(1, (t - 3.5) / 0.45); cx = (tx0 + tx1) / 2
            fade_c = lambda c: tuple(int(255 - (255 - x) * a) for x in c)
            d.text((cx - 70, GH / 2), k, font=ImageFont.truetype(FC, int(80 + 30 * a)), fill=fade_c(RED), anchor="mm")
            d.text((cx + 10, GH / 2 - 40), hv, font=ImageFont.truetype(FV, 36), fill=fade_c(TEAL))
            d.text((cx + 10, GH / 2 + 6), (m[:30] + "…") if len(m) > 31 else m, font=ImageFont.truetype(FM, 22), fill=fade_c(INK))
            if jlpt: d.text((cx + 10, GH / 2 + 38), jlpt, font=ImageFont.truetype(FM, 16), fill=fade_c((120, 120, 120)))
        frames.append(im)
    pal = frames[len(frames) // 2].quantize(colors=128, method=Image.MEDIANCUT)
    q = [f.quantize(palette=pal, dither=Image.Dither.NONE) for f in frames]
    tmp = f"/content/{no:04d}.gif"   # ghi tạm rồi chép sang Drive -> không để lại file GIF dở
    q[0].save(tmp, save_all=True, append_images=q[1:], duration=int(1000 / FPS), loop=0, optimize=True)
    subprocess.run(["cp", tmp, f"{OUT}/gif/{no:04d}.gif"], check=True)

# thẻ đã có mp4 (phiên trước bị ngắt giữa chừng) nhưng chưa có GIF -> ghép luôn
for no in CARDS:
    if os.path.exists(f"{OUT}/mp4/{no:04d}.mp4") and not done(no):
        make_gif(no); print("GIF (bù)", no)

## 5. Tạo video + GIF cho từng thẻ

Mỗi thẻ xong là lưu ngay vào Drive. Bị ngắt giữa chừng → mở lại notebook, **Run all**, nó tự bỏ qua thẻ đã xong.

In [ ]:
import gc, time, shutil
TODO = [no for no in TODO if not done(no)]
mp4 = lambda no: f"{OUT}/mp4/{no:04d}.mp4"

def finish(no, tmp, t0, i):
    shutil.copy(tmp, mp4(no)); make_gif(no)
    dt_ = time.time() - t0; left = len(TODO) - i - 1
    print(f"✅ {no} {KJ[no][1]}  {dt_ / 60:.1f} phút | còn {left} thẻ lần này ≈ {left * dt_ / 3600:.1f} giờ")

if not TODO:
    print("Không còn thẻ nào cần làm 🎉")
elif ENGINE == "wan":
    %cd /content/Wan2.2
    for i, no in enumerate(TODO):
        if os.path.exists(mp4(no)): continue
        t0 = time.time(); tmp = f"/content/{no:04d}.mp4"
        try:
            img = prep_image(no)
            subprocess.run(["python", "generate.py", "--task", "ti2v-5B", "--size", f"{W}*{H}",
                            "--ckpt_dir", "/content/Wan2.2-TI2V-5B", "--offload_model", "True", "--convert_model_dtype", "--t5_cpu",
                            "--frame_num", str(FRAMES), "--sample_steps", str(STEPS),
                            "--image", img, "--prompt", prompt(no), "--save_file", tmp], check=True)
            finish(no, tmp, t0, i)
        except Exception as e:
            print(f"⚠️ {no}: {e}")
else:
    from transformers import T5EncoderModel, T5TokenizerFast
    from diffusers import LTXImageToVideoPipeline
    from diffusers.utils import export_to_video
    M, dt = "/content/LTX-Video-0.9.1", torch.bfloat16

    # 1) mã hoá câu mô tả bằng T5 (giữ kết quả trên CPU), rồi giải phóng T5
    tok = T5TokenizerFast.from_pretrained("google-t5/t5-base", model_max_length=128)   # cùng bộ từ vựng T5, khỏi lỗi spiece.model
    t5 = T5EncoderModel.from_pretrained(M, subfolder="text_encoder", torch_dtype=dt, device_map="cuda").eval()
    @torch.no_grad()
    def enc(p):
        t = tok(p, padding="max_length", max_length=128, truncation=True, return_tensors="pt").to("cuda")
        return t5(t.input_ids, attention_mask=t.attention_mask)[0].to(dt).cpu(), t.attention_mask.bool().cpu()
    EMB = {no: enc(prompt(no)) for no in TODO}
    NEG_EMB = [x.cuda() for x in enc(NEG)]
    del t5; gc.collect(); torch.cuda.empty_cache()

    # 2) nạp LTX (transformer + VAE)
    pipe = LTXImageToVideoPipeline.from_pretrained(M, text_encoder=None, tokenizer=None, torch_dtype=dt)
    if SMALL_GPU:
        pipe.enable_model_cpu_offload()   # chỉ đưa phần đang chạy lên GPU -> chừa chỗ cho attention
    else:
        pipe.to("cuda")
    pipe.vae.enable_tiling()

    # 3) từng thẻ: ảnh -> video -> GIF -> Drive
    for i, no in enumerate(TODO):
        t0 = time.time(); tmp = f"/content/{no:04d}.mp4"
        try:
            pe, pm = (x.cuda() for x in EMB.pop(no))
            fr = pipe(image=Image.open(prep_image(no)), prompt_embeds=pe, prompt_attention_mask=pm,
                      negative_prompt_embeds=NEG_EMB[0], negative_prompt_attention_mask=NEG_EMB[1],
                      width=W, height=H, num_frames=FRAMES, num_inference_steps=STEPS,
                      generator=torch.Generator("cuda").manual_seed(42)).frames[0]
            export_to_video(fr, tmp, fps=24)
            finish(no, tmp, t0, i)
        except torch.cuda.OutOfMemoryError:
            print(f"⚠️ {no}: hết bộ nhớ GPU — bỏ qua"); gc.collect(); torch.cuda.empty_cache()
        except Exception as e:
            print(f"⚠️ {no}: {e}")

n_done = sum(map(done, CARDS))
print(f"\nĐã xong {n_done}/{len(CARDS)} thẻ. Còn lại → Run all lần nữa (tự làm tiếp).")

## 6. Xem vài GIF mới + (tuỳ chọn) tải zip về máy

Tất cả đã nằm trong Drive `MyDrive/kanji_anim/`. Ô dưới chỉ để xem nhanh / tải về.

In [ ]:
gifs = sorted(glob.glob(f"{OUT}/gif/*.gif"), key=os.path.getmtime)
print(len(gifs), "GIF trong", OUT)
for g in gifs[-3:]: display(IPImage(g))

DOWNLOAD_ZIP = False   # True -> nén cả thư mục gif tải về máy (có thể lớn)
if DOWNLOAD_ZIP and gifs:
    shutil.make_archive("/content/kanji_gif", "zip", f"{OUT}/gif")
    from google.colab import files
    files.download("/content/kanji_gif.zip")